# 01. 영상 데이터 다루기 기초 - 프레임 샘플링

Food101에서는 입력이 이미지 한 장(H, W, C)이었다. 영상은 여기에 **시간(T)** 축이 하나 더 붙는다.

영상 분류 모델(3D CNN)에 넣는 입력 단위는 보통 영상 전체가 아니라, 영상에서 일정 개수의 프레임을
뽑아 쌓은 **클립(clip)** 이다. 예를 들어 10초짜리 영상에서 16장만 균등한 간격으로 뽑아
`(T=16, H=112, W=112, C=3)` 텐서 하나로 만드는 식이다.

이 노트북에서는:
1. 학습에 쓸 작은 영상 데이터셋(UCF101의 일부 클래스)을 전체 다운로드 없이 필요한 파일만 받아오고
2. OpenCV로 영상을 열어 프레임 수 / FPS / 해상도 같은 기본 정보를 확인하고
3. 균등 샘플링으로 클립을 뽑아 그리드로 시각화해본다.


## 1-1. 왜 UCF101의 '일부'만 받는가

UCF101은 101개 동작 클래스, 총 13,320개 영상(약 6.5GB)으로 이루어진 표준 행동 인식 데이터셋이다.
지금 목표는 "영상 데이터를 다루고 파인튜닝해보는 방법"을 익히는 것이지 101개 클래스를 다 맞히는 게
아니므로, RTX 2060(6GB) 환경에 맞게 **5개 클래스 + 클래스당 소수의 영상**만 받는다.

전체 zip을 내려받지 않고, `remotezip` 라이브러리로 원격 zip 파일의 목차만 읽은 뒤
필요한 파일만 HTTP Range 요청으로 받아온다 (TensorFlow의 공식 UCF101 튜토리얼과 같은 방식).

요리 로봇 쪽 관심사와 살짝 겹치는 `CuttingInKitchen`, `Mixing`, `PizzaTossing` 을 포함해서
총 5개 클래스를 골랐다 (나머지 2개는 비교용으로 전혀 다른 동작).

In [ ]:
# 필요한 패키지 설치 (최초 1회)
# %pip install remotezip opencv-python torch torchvision matplotlib tqdm


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from video_utils import sample_frames_uniform  # 같은 폴더의 공용 유틸

DATA_DIR = Path("data/UCF101_subset")
ZIP_URL = "https://storage.googleapis.com/thumos14_files/UCF101_videos.zip"

# 비교적 적은 수의 영상만 있는 클래스들 위주로 선택 (다운로드량 절약)
CLASSES = ["CuttingInKitchen", "Mixing", "PizzaTossing", "ApplyEyeMakeup", "JugglingBalls"]
FILES_PER_CLASS = {"train": 12, "val": 4, "test": 4}  # 클래스당 영상 수


## 1-2. 원격 zip 목차 확인하기

`RemoteZip`으로 열면 파일 전체를 받지 않고도 `namelist()`로 안에 어떤 파일들이 있는지 볼 수 있다.

In [ ]:
from remotezip import RemoteZip

with RemoteZip(ZIP_URL) as zf:
    all_files = [f for f in zf.namelist() if f.endswith(".avi")]

print("전체 영상 파일 수:", len(all_files))
print(all_files[:5])


파일명이 `UCF101/v_{클래스명}_g{그룹}_c{클립}.avi` 형태이므로, 클래스명만 뽑아서
클래스별 파일 목록 딕셔너리를 만든다.

In [ ]:
import re
from collections import defaultdict


def get_class_name(filename: str) -> str:
    # 'UCF101/v_CuttingInKitchen_g01_c01.avi' -> 'CuttingInKitchen'
    return filename.split("/")[-1].split("_")[1]


files_by_class = defaultdict(list)
for f in all_files:
    files_by_class[get_class_name(f)].append(f)

for c in CLASSES:
    print(f"{c}: {len(files_by_class[c])}개 영상")


## 1-3. 필요한 영상만 다운로드해서 train/val/test로 나누기

같은 그룹(`g01`, `g02`, ...)에서 찍힌 영상들은 서로 비슷해서 같은 split에 몰아넣는 게 안전하지만,
여기서는 학습 방법 익히기가 목적이므로 단순하게 순서대로 잘라서 나눈다.

In [ ]:
import shutil

if DATA_DIR.exists():
    print(f"{DATA_DIR} 이미 존재함 - 다운로드 생략")
else:
    with RemoteZip(ZIP_URL) as zf:
        for cls in CLASSES:
            cls_files = files_by_class[cls]
            cursor = 0
            for split, n in FILES_PER_CLASS.items():
                split_dir = DATA_DIR / split / cls
                split_dir.mkdir(parents=True, exist_ok=True)
                for name in cls_files[cursor : cursor + n]:
                    zf.extract(name, path=split_dir.parent.parent)  # UCF101/xxx.avi 형태로 풀림
                    extracted = split_dir.parent.parent / name
                    extracted.rename(split_dir / extracted.name)
                cursor += n
            print(f"{cls}: 다운로드 완료")
    # 비어버린 중간 폴더(UCF101/) 정리
    leftover = DATA_DIR / "UCF101"
    if leftover.exists():
        shutil.rmtree(leftover)


In [ ]:
for split in ["train", "val", "test"]:
    for cls in CLASSES:
        n = len(list((DATA_DIR / split / cls).glob("*.avi")))
        print(f"{split:>5} / {cls:<18}: {n}개")


## 1-4. 영상 기본 정보 확인

Food101에서 `image.size`를 확인했던 것처럼, 영상도 프레임 수 / FPS / 해상도를 먼저 확인한다.

In [ ]:
import cv2

sample_path = next((DATA_DIR / "train" / CLASSES[0]).glob("*.avi"))
cap = cv2.VideoCapture(str(sample_path))
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

print(f"{sample_path.name}")
print(f"프레임 수: {n_frames}, FPS: {fps:.1f}, 해상도: {w}x{h}, 길이: {n_frames/fps:.1f}초")


## 1-5. 균등 샘플링으로 클립 뽑아서 시각화

`video_utils.sample_frames_uniform`이 영상 전체 길이에서 `num_frames`장을 균등 간격으로
뽑아준다. 영상마다 길이가 다르더라도 항상 고정된 프레임 수의 클립을 얻을 수 있다.

In [ ]:
import matplotlib.pyplot as plt

clip = sample_frames_uniform(sample_path, num_frames=16, size=112)
print("클립 shape:", clip.shape, clip.dtype)  # (T, H, W, C)

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(clip[i])
    ax.set_title(f"t={i}", fontsize=8)
    ax.axis("off")
fig.suptitle(f"{sample_path.parent.name} - 균등 샘플링된 16프레임")
plt.tight_layout()
plt.show()


다음 노트북(`02_pytorch_video_dataset.ipynb`)에서는 이 샘플링 함수를 PyTorch `Dataset`으로
감싸서 `DataLoader`로 배치 단위로 불러오는 것까지 해본다.